**0.1 Import Modules**

In [1]:
!pip install -q git+https://github.com/tensorflow/docs
!pip install imutils

ERROR: Operation cancelled by user
^C


In [2]:
from tensorflow_docs.vis import embed
from tensorflow import keras
from imutils import paths

import matplotlib.pyplot as plt
import tensorflow as tf
import pandas as pd
import numpy as np
import imageio
from IPython.display import Image, display
import cv2
import os

I0000 00:00:1777299267.832587   35536 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1777299267.875875   35536 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1777299268.933040   35536 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


**0.2 Get Data**

In [3]:
drive_path = "./videos"
train_df = pd.read_csv(f"{drive_path}/train.csv")
test_df = pd.read_csv(f"{drive_path}/test.csv")

print(f"Total videos for training: {len(train_df)}")
print(f"Total videos for testing: {len(test_df)}")

train_df.sample(10)

Total videos for training: 300
Total videos for testing: 75


,video_name,tag
63,Basketball/v_Basketball_g16_c06.avi,Basketball
286,Archery/v_Archery_g25_c04.avi,Archery
225,ApplyEyeMakeup/v_ApplyEyeMakeup_g22_c01.avi,ApplyEyeMakeup
90,BaseballPitch/v_BaseballPitch_g10_c03.avi,BaseballPitch
109,BaseballPitch/v_BaseballPitch_g23_c01.avi,BaseballPitch
240,ApplyLipstick/v_ApplyLipstick_g07_c02.avi,ApplyLipstick
111,BaseballPitch/v_BaseballPitch_g13_c05.avi,BaseballPitch
234,ApplyEyeMakeup/v_ApplyEyeMakeup_g05_c02.avi,ApplyEyeMakeup
268,ApplyLipstick/v_ApplyLipstick_g17_c03.avi,ApplyLipstick
27,BenchPress/v_BenchPress_g21_c01.avi,BenchPress


**0.2.1 Set Hyperparameters**

In [4]:
IMG_SIZE = 224
BATCH_SIZE = 64
EPOCHS = 100

MAX_SEQ_LENGTH = 20
NUM_FEATURES = 2048

**0.3 Create Helper Functions**

In [5]:
def crop_center_square(frame):
  y, x = frame.shape[0:2]
  min_dim = min(y, x)
  start_x = (x // 2)- (min_dim // 2)
  start_y = (y // 2)- (min_dim // 2)
  return frame[start_y : start_y + min_dim, start_x : start_x + min_dim]
def load_video(path, max_frames=0, resize=(IMG_SIZE, IMG_SIZE)):
  cap = cv2.VideoCapture(path)
  frames = []
  try:
    while True:
      ret, frame = cap.read()
      if not ret:
        break
      frame = crop_center_square(frame)
      frame = cv2.resize(frame, resize)
      frame = frame[:, :, [2, 1, 0]]
      frames.append(frame)
      if len(frames) == max_frames:
        break
  finally:
    cap.release()
  return np.array(frames)

In [6]:
def build_feature_extractor():
  feature_extractor = keras.applications.InceptionV3(
      weights="imagenet",
      include_top=False,
      pooling="avg",
      input_shape=(IMG_SIZE, IMG_SIZE, 3),
  )
  preprocess_input = keras.applications.inception_v3.preprocess_input

  inputs = keras.Input((IMG_SIZE, IMG_SIZE, 3))
  preprocessed = preprocess_input(inputs)

  outputs = feature_extractor(preprocessed)
  return keras.Model(inputs, outputs, name="feature_extractor")

feature_extractor = build_feature_extractor()

I0000 00:00:1777299269.900088   35536 gpu_device.cc:2043] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 749 MB memory:  -> device: 0, name: NVIDIA GeForce GTX 1650 with Max-Q Design, pci bus id: 0000:01:00.0, compute capability: 7.5


In [7]:
label_processor = keras.layers.StringLookup(num_oov_indices=0, vocabulary=np.unique(train_df["tag"]))

print(label_processor.get_vocabulary())

[np.str_('ApplyEyeMakeup'), np.str_('ApplyLipstick'), np.str_('Archery'), np.str_('BabyCrawling'), np.str_('BalanceBeam'), np.str_('BandMarching'), np.str_('BaseballPitch'), np.str_('Basketball'), np.str_('BasketballDunk'), np.str_('BenchPress')]


In [8]:
def prepare_all_videos(df, root_dir):
  num_samples = len(df)
  video_paths = df["video_name"].values.tolist()
  labels = df["tag"].values
  labels = label_processor(labels[..., None]).numpy()

  # `frame_masks` and `frame_features` are what we will feed to our sequence model.
  # `frame_masks` will contain a bunch of booleans denoting if a timestep is
  # masked with padding or not.
  frame_masks = np.zeros(shape=(num_samples, MAX_SEQ_LENGTH), dtype="bool")
  frame_features = np.zeros(
      shape=(num_samples, MAX_SEQ_LENGTH, NUM_FEATURES), dtype="float32"
  )

  # For each video.
  for idx, path in enumerate(video_paths):
    # Gather all its frames and add a batch dimension.
    frames = load_video(os.path.join(root_dir, path))
    frames = frames[None, ...]

    # Initialize placeholders to store the masks and features of the current video.
    temp_frame_mask = np.zeros(shape=(1, MAX_SEQ_LENGTH,), dtype="bool")
    temp_frame_features = np.zeros(
        shape=(1, MAX_SEQ_LENGTH, NUM_FEATURES), dtype="float32"
    )

    # Extract features from the frames of the current video.
    for i, batch in enumerate(frames):
      video_length = batch.shape[0]
      length = min(MAX_SEQ_LENGTH, video_length)
      for j in range(length):
        temp_frame_features[i, j, :] = feature_extractor.predict(
            batch[None, j, :])

      temp_frame_mask[i, :length] = 1 # 1 = not masked, 0 = masked
    frame_features[idx,] = temp_frame_features.squeeze()
    frame_masks[idx,] = temp_frame_mask.squeeze()
  return (frame_features, frame_masks), labels

train_data, train_labels = prepare_all_videos(train_df, f"{drive_path}/Materi Ngajar/Minggu 5/train")
test_data, test_labels = prepare_all_videos(test_df, f"{drive_path}/Materi Ngajar/Minggu 5/test")

print(f"Frame features in train set: {train_data[0].shape}")
print(f"Frame masks in train set: {train_data[1].shape}")

Frame features in train set: (300, 20, 2048)
Frame masks in train set: (300, 20)


In [9]:
# Utility for our sequence model.
def get_sequence_model():
  class_vocab = label_processor.get_vocabulary()

  frame_features_input = keras.Input((MAX_SEQ_LENGTH, NUM_FEATURES))
  mask_input = keras.Input((MAX_SEQ_LENGTH,), dtype="bool")

  # Refer to the following tutorial to understand the significance of using `mask`:
  # https://keras.io/api/layers/recurrent_layers/gru/
  x = keras.layers.GRU(16, return_sequences=True, use_cudnn=False)(
      frame_features_input, mask=mask_input
  )
  x = keras.layers.GRU(8, use_cudnn=False)(x)
  x = keras.layers.Dropout(0.4)(x)
  x = keras.layers.Dense(8, activation="relu")(x)
  output = keras.layers.Dense(len(class_vocab), activation="softmax")(x)

  rnn_model = keras.Model([frame_features_input, mask_input], output)

  rnn_model.compile(
      loss="sparse_categorical_crossentropy", optimizer="adam", metrics=["accuracy"]
  )
  return rnn_model

# Utility for running experiments.
def run_experiment():
  seq_model = get_sequence_model()
  history = seq_model.fit(
    [train_data[0], train_data[1]],
    train_labels,
    validation_split=0.3,
    epochs=50,
  )
  _, accuracy = seq_model.evaluate([test_data[0], test_data[1]], test_labels)
  print(f"Test accuracy: {round(accuracy * 100, 2)}%")

  return history, seq_model

_, sequence_model = run_experiment()

Epoch 1/50


I0000 00:00:1777299273.936598   35750 service.cc:153] XLA service 0x7e6ffc0052a0 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1777299273.936620   35750 service.cc:161]   StreamExecutor [0]: NVIDIA GeForce GTX 1650 with Max-Q Design, Compute Capability 7.5 (Driver: 13.0.0; Runtime: 12.9.0; Toolkit: 12.5.0; DNN: 9.20.0)
I0000 00:00:1777299273.985743   35750 dump_mlir_util.cc:269] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
I0000 00:00:1777299274.327801   35750 cuda_dnn.cc:461] Loaded cuDNN version 92000
I0000 00:00:1777299274.418315   35750 dot_merger.cc:481] Merging Dots in computation: functional_1_gru_1_2_while_body_6606_grad_6863_const_0__.30.clone.clone.clone.clone.clone.clone.clone.clone
I0000 00:00:1777299274.418403   35750 dot_merger.cc:481] Merging Dots in computation: functional_1_gru_1_while_body_6406_grad_7450_const_0__.38.clone.clone.clone.clone.clone.clone.clone
I0000 00:00:

1/7 ━━━━━━━━━━━━━━━━━━━━ 27s 5s/step - accuracy: 0.0000e+00 - loss: 2.3026

I0000 00:00:1777299276.673547   35750 device_compiler.h:208] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.
I0000 00:00:1777299277.254283   35750 dot_merger.cc:481] Merging Dots in computation: functional_1_gru_1_2_while_body_6606_grad_6863_const_0__.30.clone.clone.clone.clone.clone.clone.clone.clone
I0000 00:00:1777299277.254371   35750 dot_merger.cc:481] Merging Dots in computation: functional_1_gru_1_while_body_6406_grad_7450_const_0__.38.clone.clone.clone.clone.clone.clone.clone
I0000 00:00:1777299277.254428   35750 dot_merger.cc:481] Merging Dots in computation: a_inference_one_step_on_data_8434__.39


7/7 ━━━━━━━━━━━━━━━━━━━━ 9s 721ms/step - accuracy: 0.1190 - loss: 2.3011 - val_accuracy: 0.0000e+00 - val_loss: 2.3110
Epoch 2/50
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step - accuracy: 0.1429 - loss: 2.2976 - val_accuracy: 0.0000e+00 - val_loss: 2.3191
Epoch 3/50
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step - accuracy: 0.1429 - loss: 2.2944 - val_accuracy: 0.0000e+00 - val_loss: 2.3270
Epoch 4/50
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step - accuracy: 0.1429 - loss: 2.2911 - val_accuracy: 0.0000e+00 - val_loss: 2.3347
Epoch 5/50
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step - accuracy: 0.1429 - loss: 2.2879 - val_accuracy: 0.0000e+00 - val_loss: 2.3424
Epoch 6/50
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step - accuracy: 0.1381 - loss: 2.2848 - val_accuracy: 0.0000e+00 - val_loss: 2.3502
Epoch 7/50
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step - accuracy: 0.1238 - loss: 2.2816 - val_accuracy: 0.0000e+00 - val_loss: 2.3580
Epoch 8/50
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step - accuracy: 0.1381 - loss: 2.2785 - val_accuracy: 0.0000e+

In [10]:
from tensorflow.keras.layers import LSTM

def get_sequence_model():
  class_vocab = label_processor.get_vocabulary()

  frame_features_input = keras.Input((MAX_SEQ_LENGTH, NUM_FEATURES))
  mask_input = keras.Input((MAX_SEQ_LENGTH,), dtype="bool")

  x = LSTM(16, return_sequences=True, use_cudnn=False)(
  frame_features_input, mask=mask_input
  )
  x = LSTM(8 ,use_cudnn=False)(x)
  x = keras.layers.Dropout(0.4)(x)
  x = keras.layers.Dense(8, activation="relu")(x)
  output = keras.layers.Dense(len(class_vocab), activation="softmax")(x)

  lstm = keras.Model([frame_features_input, mask_input], output)

  lstm.compile(
      loss="sparse_categorical_crossentropy", optimizer="adam", metrics=["accuracy"]
  )
  return lstm

# Utility for running experiments.
def run_experiment():
  seq_model = get_sequence_model()
  history = seq_model.fit(
    [train_data[0], train_data[1]],
    train_labels,
    validation_split=0.3,
    epochs=EPOCHS,
  )
  _, accuracy = seq_model.evaluate([test_data[0], test_data[1]], test_labels)
  print(f"Test accuracy: {round(accuracy * 100, 2)}%")

  return history, seq_model

_, sequence_model = run_experiment()

Epoch 1/100


I0000 00:00:1777299290.718433   35748 dot_merger.cc:481] Merging Dots in computation: functional_1_1_lstm_1_2_while_body_15414_grad_15660_const_0__.28.clone.clone.clone.clone.clone.clone.clone.clone
I0000 00:00:1777299290.718546   35748 dot_merger.cc:481] Merging Dots in computation: functional_1_1_lstm_1_while_body_15241_grad_16197_const_0__.34.clone.clone.clone.clone.clone.clone.clone
I0000 00:00:1777299290.718610   35748 dot_merger.cc:481] Merging Dots in computation: a_inference_one_step_on_data_17130__.35


1/7 ━━━━━━━━━━━━━━━━━━━━ 20s 3s/step - accuracy: 0.0000e+00 - loss: 2.3026

I0000 00:00:1777299292.673182   35748 dot_merger.cc:481] Merging Dots in computation: functional_1_1_lstm_1_2_while_body_15414_grad_15660_const_0__.28.clone.clone.clone.clone.clone.clone.clone.clone
I0000 00:00:1777299292.673303   35748 dot_merger.cc:481] Merging Dots in computation: functional_1_1_lstm_1_while_body_15241_grad_16197_const_0__.34.clone.clone.clone.clone.clone.clone.clone
I0000 00:00:1777299292.673371   35748 dot_merger.cc:481] Merging Dots in computation: a_inference_one_step_on_data_17130__.35


7/7 ━━━━━━━━━━━━━━━━━━━━ 7s 534ms/step - accuracy: 0.1143 - loss: 2.3012 - val_accuracy: 0.0000e+00 - val_loss: 2.3109
Epoch 2/100
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - accuracy: 0.1429 - loss: 2.2977 - val_accuracy: 0.0000e+00 - val_loss: 2.3192
Epoch 3/100
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - accuracy: 0.0905 - loss: 2.2942 - val_accuracy: 0.0000e+00 - val_loss: 2.3273
Epoch 4/100
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - accuracy: 0.1429 - loss: 2.2909 - val_accuracy: 0.0000e+00 - val_loss: 2.3354
Epoch 5/100
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step - accuracy: 0.1190 - loss: 2.2876 - val_accuracy: 0.0000e+00 - val_loss: 2.3434
Epoch 6/100
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - accuracy: 0.1286 - loss: 2.2844 - val_accuracy: 0.0000e+00 - val_loss: 2.3514
Epoch 7/100
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - accuracy: 0.1429 - loss: 2.2811 - val_accuracy: 0.0000e+00 - val_loss: 2.3593
Epoch 8/100
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - accuracy: 0.1429 - loss: 2.2780 - val_accuracy: 0

In [11]:
def prepare_single_video(frames):
  frames = frames[None, ...]
  frame_mask = np.zeros(shape=(1, MAX_SEQ_LENGTH,), dtype="bool")
  frame_features = np.zeros(shape=(1, MAX_SEQ_LENGTH, NUM_FEATURES), dtype="float32")

  for i, batch in enumerate(frames):
    video_length = batch.shape[0]
    length = min(MAX_SEQ_LENGTH, video_length)
    for j in range(length):
      frame_features[i, j, :] = feature_extractor.predict(batch[None, j, :])
      frame_mask[i, :length] = 1 # 1 = not masked, 0 = masked

  return frame_features, frame_mask

def sequence_prediction(path):
  class_vocab = label_processor.get_vocabulary()

  frames = load_video(os.path.join("test", path))
  frame_features, frame_mask = prepare_single_video(frames)
  probabilities = sequence_model.predict([frame_features, frame_mask])[0]

  for i in np.argsort(probabilities)[::-1]:
    print(f" {class_vocab[i]}: {probabilities[i] * 100:5.2f}%")
  return frames

def to_gif(images):
  converted_images = np.clip(images * 255, 0, 255).astype(np.uint8)
  imageio.mimsave('animation.gif', converted_images, fps=25)
  return embed.embed_file('animation.gif')

**0.4 Test Video**

In [12]:
test_video = np.random.choice(test_df["video_name"].values.tolist())
print(f"Test video path: {test_video}")
test_frames = sequence_prediction(test_video)
#to_gif(test_frames[:MAX_SEQ_LENGTH])

Test video path: BabyCrawling/v_BabyCrawling_g04_c01.avi
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 750ms/step
 BabyCrawling: 12.32%
 Basketball: 12.23%
 BandMarching: 12.18%
 BenchPress: 12.17%
 BasketballDunk: 12.10%
 BalanceBeam: 12.09%
 BaseballPitch: 12.08%
 Archery:  4.94%
 ApplyLipstick:  4.94%
 ApplyEyeMakeup:  4.94%
